![Logo 1](https://git.wmi.amu.edu.pl/AITech/Szablon/raw/branch/master/Logotyp_AITech1.jpg)
<div class="alert alert-block alert-info">
<h1> Komputerowe wspomaganie tłumaczenia </h1>
<h2> 15. <i>Korekta gramatyczna</i> [laboratoria]</h2> 
<h3>Rafał Jaworski (2021)</h3>
</div>

![Logo 2](https://git.wmi.amu.edu.pl/AITech/Szablon/raw/branch/master/Logotyp_AITech2.jpg)

Ostatnią z omawianych przez nas technik stosowaną podczas wspomagania tłumaczenia jest korekta gramatyczna. Automatyczna korekta gramatyczna tekstu to ambitne zadanie odnalezienia możliwych błędów niezwiązanych bezpośrednio z pisownią. Są to między innymi:
* błędy gramatyczne
* źle użyte słowa
* złe połączenia wyrazowe
* błędy interpunkcyjne
* kolokwializmy
* redundancja (np. "tylko i wyłącznie")

Warto zwrócić uwagę, iż systemy do korekcji gramatycznej można traktować jako klasyfikatory binarne. Przyjmijmy, że odpowiedź pozytywna korektora to wykrycie błędu w tekście, natomiast negatywna - brak błędu. Wówczas rozróżniamy dwa typy pomyłek: false positive oraz false negative. False positive to tzw. "fałszywy alarm" - zbyt duża ich ilość spowoduje wydłużenie czasu pracy użytkownika przez konieczność analizowania zgłoszeń, które w istocie błędami nie są. Co jednak jeszcze gorsze, zbyt duża ilość false positives powoduje spadek zaufania użytkownika do systemu oraz drastyczny spadek satysfakcji z używania systemu. Te drugie błędy - false negatives - to z kolei faktyczne błędy w tekście, które nie zostały wyłapane przez system korekty. Stare polskie porzekadło głosi, że "czego oko nie widzi, tego sercu nie żal". Niestety jednak problem pojawia się, kiedy dostrzeże to jakieś inne oko... Wysoka liczba false negatives wprawdzie skraca czas korekty (sic!), ale odbywa się to kosztem jakości całego procesu. Idealnie zatem byłoby zminimalizować zarówno liczbę false positives, jak i false negatives. Jak jednak łatwo się domyślić, nie jest to zawsze możliwe. Korektor gramatyczny, który jest bardzo restrykcyjny i raportuje wiele błędów, będzie miał tendencję do popełniania false positives. Natomiast korektor bardziej pobłażliwy niechybnie popełni wiele false negatives. Co zatem jest ważniejsze? Praktyka wskazuje, że oba parametry mają podobną wagę, ale jednak odrobinę ważniejsze jest powstrzymanie się od false positives.

Do najbardziej popularnych narzędzi wspomagających korektę gramatyczną tekstu należą Grammarly oraz LanguageTool. Na dzisiejszych zajęciach zajmiemy się tym drugim. LanguageTool został pierwotnie napisany jako praca dyplomowa Daniela Nabera, a następnie intensywnie rozwijany wspólnie z Marcinem Miłkowskim. Aż do dziś projekt jest ciągle rozwijany, zwiększana jest liczba obsługiwanych języków oraz dokładność działania.

LanguageTool jest systemem opartym na regułach. W dobie wszechobecnej sztucznej inteligencji opartej na uczeniu maszynowym rozwiązanie to wydaje się nieco przestarzałe. Jednak to właśnie reguły stanowią o sile LanguageToola - pozwalają one na zwiększenie precyzji korektora poprzez minimalizację false positives. Warto wspomnieć, iż liczne reguły LanguageToola dotyczą również korekty pisowni. Czyni to z LanguageToola kompletne narzędzie do korekty tekstu. Polecam przejrzenie zestawu reguł LanguageToola dla języka angielskiego: https://community.languagetool.org/rule/list?lang=en

Czas uruchomić to narzędzie. Skorzystajmy z Pythona.

Następnie możemy użyć LanguageToola w programie Pythonowym: (przykład zaczerpnięty z oficjalnego tutoriala: https://pypi.org/project/language-tool-python/)

In [1]:
import language_tool_python
import pprint
tool = language_tool_python.LanguageTool('en-US') 

text = 'A sentence with a error in the Hitchhiker’s Guide tot he Galaxy'

pp = pprint.PrettyPrinter(depth=2)
errors = tool.check(text)
pp.pprint(errors)

ModuleNotFoundError: No java install detected. Please install java to use language-tool-python.

Przeanalizujmy format zwracanego wyniku. Otrzymujemy listę obiektów Match - zawiadomień o potencjalnym błędzie. Razem z każdym błędem otrzymujemy m.in. identyfikator użytej reguły, opis błędu, rekomendancję poprawy, kontekst.

### Ćwiczenie 1: Użyj LanguageTool do znalezienia jak największej liczby prawdziwych błędów na swoim ulubionym portalu internetowym. Skorzystaj z poznanych wcześniej technik web scrapingu. Uwaga - LanguageTool najprawdopodobniej oznaczy nazwy własne jako literówki - ten typ błędu nie powinien być brany pod uwagę.

In [ ]:
import re
import requests
from bs4 import BeautifulSoup


IGNORED_RULES = {
    "MORFOLOGIK_RULE_EN_US",  # catches many proper names as spelling mistakes
    "MORFOLOGIK_RULE_EN_GB",
}


def _get_tool(language="en-US"):
    """Create LanguageTool lazily and fall back to the public API if Java is unavailable."""
    import language_tool_python

    try:
        return language_tool_python.LanguageTool(language)
    except ModuleNotFoundError as exc:
        if "java" not in str(exc).lower():
            raise
        return language_tool_python.LanguageToolPublicAPI(language)


def _visible_page_text(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript", "svg", "header", "footer", "nav", "form"]):
        tag.decompose()

    chunks = [part.strip() for part in soup.stripped_strings]
    return "\n".join(chunk for chunk in chunks if len(chunk) > 2)


def _looks_like_proper_name(match):
    """Reduce false positives caused by names, brands and all-caps abbreviations."""
    token = match.context[match.offsetInContext:match.offsetInContext + match.errorLength]
    if not token:
        return False
    return (
        token.isupper()
        or (token[0].isupper() and token.lower() not in {"i"})
        or re.search(r"[A-Z][a-z]+(?:[A-Z][a-z]+)+", token) is not None
    )


def _match_to_dict(match, source=None):
    data = {
        "rule_id": match.ruleId,
        "message": match.message,
        "suggestions": list(match.replacements[:5]),
        "context": match.context,
        "offset": match.offset,
        "error_length": match.errorLength,
        "category": getattr(match, "category", None),
    }
    if source is not None:
        data["source"] = source
    return data


def find_errors(website_url, language="en-US", limit=50):
    """
    Download text from a web page and return likely real LanguageTool errors.

    Spelling-only alerts for proper names are ignored, because news portals contain
    many people, institution and product names that LanguageTool reports as typos.
    """
    response = requests.get(
        website_url,
        timeout=15,
        headers={"User-Agent": "Mozilla/5.0 (compatible; KWT-Lab15/1.0)"},
    )
    response.raise_for_status()

    text = _visible_page_text(response.text)
    tool = _get_tool(language)
    results = []

    for match in tool.check(text):
        if match.ruleId in IGNORED_RULES and _looks_like_proper_name(match):
            continue
        if getattr(match, "ruleIssueType", None) == "misspelling" and _looks_like_proper_name(match):
            continue
        results.append(_match_to_dict(match, source=website_url))
        if len(results) >= limit:
            break

    return results


# Example usage:
# errors = find_errors("https://example.com", limit=10)
# pprint.pp(errors)





### Ćwiczenie 2: Napisz skrypt, który poszuka błędów w komentarzach klasy Javowej (zwykłych // oraz w javadocach). Uruchom ten skrypt na źródłach wybranego opensourcowego projektu w Javie.

In [ ]:
import re
from pathlib import Path


_COMMENT_RE = re.compile(
    r"//(?P<line>.*?$)|/\*(?P<block>.*?)\*/",
    re.MULTILINE | re.DOTALL,
)
_STRING_OR_CHAR_RE = re.compile(
    r'"(?:\\.|[^"\\])*"|\'(?:\\.|[^\'\\])*\'',
    re.DOTALL,
)


def _mask_string_literals(java_source):
    """Keep comment offsets stable while preventing // inside strings from matching."""
    return _STRING_OR_CHAR_RE.sub(lambda m: " " * (m.end() - m.start()), java_source)


def _clean_java_comment(comment):
    lines = comment.splitlines()
    cleaned = []
    for line in lines:
        line = re.sub(r"^\s*//\s?", "", line)
        line = re.sub(r"^\s*/\*+\s?", "", line)
        line = re.sub(r"\s*\*/\s?$", "", line)
        line = re.sub(r"^\s*\*\s?", "", line)
        line = re.sub(r"\{@\w+\s+([^}]*)\}", r"\1", line)
        line = re.sub(r"@(param|return|throws|exception|see|link|code|value|author|since)\b", "", line)
        line = re.sub(r"<[^>]+>", " ", line)
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            cleaned.append(line)
    return " ".join(cleaned)


def _line_number(source, offset):
    return source.count("\n", 0, offset) + 1


def extract_java_comments(java_source):
    masked = _mask_string_literals(java_source)
    comments = []

    for match in _COMMENT_RE.finditer(masked):
        raw = java_source[match.start():match.end()]
        text = _clean_java_comment(raw)
        if not text:
            continue
        comments.append(
            {
                "line": _line_number(java_source, match.start()),
                "kind": "line" if match.group("line") is not None else "block/javadoc",
                "text": text,
            }
        )

    return comments


def correct_java_grammar(java_file_path, language="en-US", limit=None):
    """Find LanguageTool grammar issues in //, /* */ and /** */ Java comments."""
    path = Path(java_file_path)
    source = path.read_text(encoding="utf-8", errors="replace")
    tool = _get_tool(language)
    results = []

    for comment in extract_java_comments(source):
        for match in tool.check(comment["text"]):
            if match.ruleId in IGNORED_RULES and _looks_like_proper_name(match):
                continue
            issue = _match_to_dict(match, source=str(path))
            issue.update(
                {
                    "line": comment["line"],
                    "comment_kind": comment["kind"],
                    "comment": comment["text"],
                }
            )
            results.append(issue)
            if limit is not None and len(results) >= limit:
                return results

    return results


# Example usage on a cloned Java project:
# all_errors = []
# for java_file in Path("/path/to/project").rglob("*.java"):
#     all_errors.extend(correct_java_grammar(java_file, limit=20))
# pprint.pp(all_errors[:20])



Przykładowe użycie funkcji dla pojedynczego pliku Java:


In [ ]:
from tempfile import NamedTemporaryFile

java_example = """
public class Demo {
    /**
     * This are a bad sentence.
     */
    // A sentence with a error in comment.
    String url = "http://example.com//this_is_not_a_comment";
}
"""

with NamedTemporaryFile("w", suffix=".java", delete=False, encoding="utf-8") as java_file:
    java_file.write(java_example)
    java_path = java_file.name

# Requires Java for local LanguageTool or access to the public LanguageTool API.
# correct_java_grammar(java_path, limit=10)
extract_java_comments(java_example)

